In [1]:
from dataclasses import replace
import tabulate

from config.experiment import ExperimentConfig
from experiments import load_config
from config.task import generate_task_configs
from utils.plot import Metric, plot_metrics_vs_perturbation, get_metrics, shorten_model_name, strip_hf_org, PlotFilter

SyntaxError: invalid syntax (configstore.py, line 25)

In [28]:
def get_config(cfg_name: str) -> ExperimentConfig:
    exp_cfg = load_config(cfg_name)

    return ExperimentConfig(
        train = replace(exp_cfg.train, save_folder = exp_cfg.train.save_folder.parent / "paper"),
        eval = replace(exp_cfg.eval, results_folder = exp_cfg.eval.results_folder.parent / "paper"),
    )

In [29]:
# exp_config = get_config("paper_perturbation_plot")

# plot_metrics_vs_perturbation(exp_config, metrics_to_show=[Metric.RETRIEVAL_ASR_TRAIN, Metric.RETRIEVAL_ASR_TEST, Metric.GENERATION_ASR_EXACT_TRAIN, Metric.GENERATION_ASR_EXACT_TEST, Metric.GENERATION_ACC_EMBED_GT_TRAIN, Metric.GENERATION_ACC_EMBED_GT_TEST], ret_topk_idx=0, gen_topk_idx=0)

In [30]:
# exp_config = get_config("perturbation_plot_targeted")
#
# plot_metrics_vs_perturbation(exp_config, metrics_to_show=[Metric.RETRIEVAL_ASR_TRAIN, Metric.RETRIEVAL_ASR_TEST,
#                                                           Metric.GENERATION_ASR_EXACT_TRAIN,
#                                                           Metric.GENERATION_ASR_EXACT_TEST,
#                                                           Metric.GENERATION_ACC_EMBED_GT_TRAIN,
#                                                           Metric.GENERATION_ACC_EMBED_GT_TEST], ret_topk_idx=0,
#                              gen_topk_idx=0)

In [31]:
def make_all_metric_table(config_name: str, metrics_to_show:list[Metric]|None=None, row_filter = None):
    exp_config = get_config(config_name)
    if metrics_to_show is None:
        metrics_to_show = [m for m in Metric]

    task_configs = generate_task_configs(exp_config, include_eval=True)

    table = []
    for task_config in task_configs:
        row = {
            "dataset": strip_hf_org(task_config.ds_name),
        }
        if not exp_config.eval.test_gpt_attack:
            row["embedder"] = shorten_model_name(task_config.model_name_embs[0]) if len(task_config.model_name_embs) == 1 else "+".join([shorten_model_name(m) for m in task_config.model_name_embs])
            if task_config.vlm:
                row["vlm"] = shorten_model_name(task_config.vlm.models[0]) if len(task_config.vlm.models) == 1 else "+".join([shorten_model_name(m) for m in task_config.vlm.models])
                if len(exp_config.train.vlm.gen_topk_list) > 1:
                    row["vlm topk"] = task_config.vlm.gen_topk
        if len(exp_config.train.emb_train_loss_type_list) > 1:
            row["emb train loss"] = task_config.emb_train_loss_type
        if len(exp_config.train.attack_mask_list) > 1:
            row["attack mask"] = task_config.attack_mask.name
        if task_config.eval_emb_name:
            row["eval emb"] = shorten_model_name(task_config.eval_emb_name)
        if task_config.eval_vlm_name:
            row["eval vlm"] = shorten_model_name(task_config.eval_vlm_name)

        if task_config.judge:
            row["judge"] = shorten_model_name(task_config.judge.model_name)
        if task_config.eval_jdg_name:
            row["eval judge"] = shorten_model_name(task_config.eval_jdg_name)
        metrics, _ = get_metrics(
            exp_config=exp_config,
            task_config=task_config,
            metrics_to_show=metrics_to_show,
        )
        row.update(metrics)
        table.append(row)
    if row_filter:
        table = [row for row in table if row_filter(row)]
    # return tabulate.tabulate(table, headers="keys", tablefmt="latex", showindex="never")
    return tabulate.tabulate(table, headers="keys", tablefmt="html", showindex="always")

In [32]:
make_all_metric_table("paper_non_targeted", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, row_filter=PlotFilter.CONDITION_SAME_MODELS)

,dataset,embedder,vlm,eval emb,eval vlm,Recall-B@1,Recall-A@1,ASR-R (test)@1,Recall-B@5,Recall-A@5,ASR-R (test)@5,ASR-G-HARD (test)@-1,SIM-G-ADV (test)@-1,SIM-G-GT (test)@-1
0,restaurant_esg_reports_beir,CLIP-L,SmolVLM,CLIP-L,SmolVLM,0.133925,0.0181818,0.636364,0.360089,0.323725,0.909091,1,1,0.0460204
1,restaurant_esg_reports_beir,CLIP-L,Qwen2.5-VL-3B,CLIP-L,Qwen2.5-VL-3B,0.133925,0.0181818,0.727273,0.360089,0.341907,0.727273,1,1,0.0460204
2,restaurant_esg_reports_beir,ColPali,SmolVLM,ColPali,SmolVLM,0,0,0,0,0,0,0.818182,0.948878,0.0570249
3,restaurant_esg_reports_beir,ColPali,Qwen2.5-VL-3B,ColPali,Qwen2.5-VL-3B,0,0,0,0,0,0,1,1,0.0460204
4,restaurant_esg_reports_beir,GME-Qwen2-VL-2B,SmolVLM,GME-Qwen2-VL-2B,SmolVLM,0.462971,0.443459,0,0.712639,0.712639,0.0909091,1,1,0.0460204
5,restaurant_esg_reports_beir,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.462971,0.443459,0,0.712639,0.712639,0.0909091,1,1,0.0460204


In [33]:
make_all_metric_table("paper_targeted_attacks_oneQ_oneA", row_filter=PlotFilter.CONDITION_SAME_MODELS, metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

,dataset,embedder,vlm,eval emb,eval vlm,ASR-R targeted@1,FPR-R targeted (test)@1,ASR-R targeted@5,FPR-R targeted (test)@5,SIM-G-ADV-POS targeted (train)@-1,SIM-G-ADV-NEG targeted (test)@-1
0,restaurant_esg_reports_beir,CLIP-L,SmolVLM,CLIP-L,SmolVLM,1,0,1,0,0.999978,0.0867446
1,restaurant_esg_reports_beir,CLIP-L,Qwen2.5-VL-3B,CLIP-L,Qwen2.5-VL-3B,1,0,1,0,0.871279,0.104722
2,restaurant_esg_reports_beir,ColPali,SmolVLM,ColPali,SmolVLM,0,0,0,0,0.0443063,0.0735199
3,restaurant_esg_reports_beir,ColPali,Qwen2.5-VL-3B,ColPali,Qwen2.5-VL-3B,0,0,0,0,0.0969508,0.0578607
4,restaurant_esg_reports_beir,GME-Qwen2-VL-2B,SmolVLM,GME-Qwen2-VL-2B,SmolVLM,1,0,1,0,0.999978,0.114039
5,restaurant_esg_reports_beir,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,1,0,1,0,0.210228,0.141643


In [ ]:
make_all_metric_table("paper_targeted_attacks_multiQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_targeted_attacks_multiQ_multiA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_judge_defence", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

In [ ]:
make_all_metric_table("paper_judge_defence_adapt", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

In [ ]:
make_all_metric_table("paper_judge_defence_targeted", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

In [ ]:
make_all_metric_table("paper_judge_defence_targeted_adapt", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

In [ ]:
make_all_metric_table("paper_copali_ab", metrics_to_show=PlotFilter.METRICS_COLPALI)

In [ ]:
make_all_metric_table("paper_copali_ab_cpoiT", metrics_to_show=PlotFilter.METRICS_COLPALI)

In [ ]:
make_all_metric_table("paper_topk_context", metrics_to_show=PlotFilter.METRICS_TOPK)

In [ ]:
make_all_metric_table("paper_topk_context_targeted", metrics_to_show=PlotFilter.METRICS_TOPK_TARGETED)

In [ ]:
make_all_metric_table("paper_defences", metrics_to_show=PlotFilter.METRICS_TOPK)

In [ ]:
make_all_metric_table("paper_targeted_defences", metrics_to_show=PlotFilter.METRICS_TOPK_TARGETED)

In [ ]:
# make_all_metric_table("mask_attack")

In [ ]:
make_all_metric_table("paper_GPT_non_targeted", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

In [ ]:
make_all_metric_table("paper_GPT_targeted_attacks_oneQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_multiA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_multi_transferability", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

In [ ]:
make_all_metric_table("paper_multi_transferability_targeted", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)